# Benchmark: A* vs D* Lite (Bản đồ Romania Baseline)
So sánh hiệu năng theo yêu cầu đồ án: Thời gian, Bộ nhớ, Số lượng Expand Node, Worst Case vs Best Case.

In [ ]:
import time
import heapq
import pandas as pd

# 1. Khởi tạo bản đồ
romania_map = {
    "roads": {
        "Arad": {
            "Zerind": 75,
            "Sibiu": 140,
            "Timisoara": 118
        },
        "Bucharest": {
            "Urziceni": 85,
            "Pitesti": 101,
            "Giurgiu": 90,
            "Fagaras": 211
        },
        "Craiova": {
            "Drobeta": 120,
            "Rimnicu Vilcea": 146,
            "Pitesti": 138
        },
        "Drobeta": {
            "Mehadia": 75,
            "Craiova": 120
        },
        "Eforie": {
            "Hirsova": 86
        },
        "Fagaras": {
            "Sibiu": 99,
            "Bucharest": 211
        },
        "Giurgiu": {
            "Bucharest": 90
        },
        "Hirsova": {
            "Urziceni": 98,
            "Eforie": 86
        },
        "Iasi": {
            "Vaslui": 92,
            "Neamt": 87
        },
        "Lugoj": {
            "Timisoara": 111,
            "Mehadia": 70
        },
        "Oradea": {
            "Zerind": 71,
            "Sibiu": 151
        },
        "Pitesti": {
            "Rimnicu Vilcea": 97,
            "Craiova": 138,
            "Bucharest": 101
        },
        "Rimnicu Vilcea": {
            "Sibiu": 80,
            "Craiova": 146,
            "Pitesti": 97
        },
        "Urziceni": {
            "Vaslui": 142,
            "Bucharest": 85,
            "Hirsova": 98
        },
        "Zerind": {
            "Arad": 75,
            "Oradea": 71
        },
        "Sibiu": {
            "Arad": 140,
            "Fagaras": 99,
            "Oradea": 151,
            "Rimnicu Vilcea": 80
        },
        "Timisoara": {
            "Arad": 118,
            "Lugoj": 111
        },
        "Mehadia": {
            "Lugoj": 70,
            "Drobeta": 75
        },
        "Vaslui": {
            "Urziceni": 142,
            "Iasi": 92
        },
        "Neamt": {
            "Iasi": 87
        }
    },
    "locations": {
        "Arad": [
            91,
            492
        ],
        "Bucharest": [
            400,
            327
        ],
        "Craiova": [
            253,
            288
        ],
        "Drobeta": [
            165,
            299
        ],
        "Eforie": [
            562,
            293
        ],
        "Fagaras": [
            305,
            449
        ],
        "Giurgiu": [
            375,
            270
        ],
        "Hirsova": [
            534,
            350
        ],
        "Iasi": [
            473,
            506
        ],
        "Lugoj": [
            165,
            379
        ],
        "Oradea": [
            131,
            571
        ],
        "Pitesti": [
            320,
            368
        ],
        "Rimnicu Vilcea": [
            233,
            410
        ],
        "Urziceni": [
            456,
            350
        ],
        "Zerind": [
            108,
            531
        ],
        "Sibiu": [
            207,
            457
        ],
        "Timisoara": [
            94,
            410
        ],
        "Mehadia": [
            168,
            339
        ],
        "Vaslui": [
            509,
            444
        ],
        "Neamt": [
            406,
            537
        ]
    }
}

def heuristic_euclidean(node, goal="Bucharest"):
    (x1, y1) = romania_map["locations"][node]
    (x2, y2) = romania_map["locations"][goal]
    return ((x1 - x2)**2 + (y1 - y2)**2)**0.5


In [ ]:
def a_star(start, goal, graph):
    open_set = []
    heapq.heappush(open_set, (0, start))
    came_from = {}
    g_score = {node: float('inf') for node in graph}
    g_score[start] = 0
    expanded_nodes = 0
    max_memory = 0
    start_time = time.perf_counter()
    while open_set:
        max_memory = max(max_memory, len(open_set))
        current_f, current = heapq.heappop(open_set)
        expanded_nodes += 1
        if current == goal: break
        for neighbor, cost in graph[current].items():
            if cost == float('inf'): continue
            tentative_g = g_score[current] + cost
            if tentative_g < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                f_score = tentative_g + heuristic_euclidean(neighbor, goal)
                heapq.heappush(open_set, (f_score, neighbor))
    return {'expanded_nodes': expanded_nodes, 'max_memory_nodes': max_memory, 'time_ms': (time.perf_counter() - start_time) * 1000}


In [ ]:
class DStarLiteSimplified:
    def __init__(self, start, goal, graph):
        self.start = start
        self.goal = goal
        self.graph = {u: {v: w for v, w in neighbors.items()} for u, neighbors in graph.items()}
        self.rhs = {node: float('inf') for node in graph}
        self.g = {node: float('inf') for node in graph}
        self.rhs[goal] = 0
        self.U = []
        self.expanded_nodes = 0
        self.max_memory = 0
        heapq.heappush(self.U, (self.calculate_key(goal), goal))
    def calculate_key(self, s):
        return (min(self.g[s], self.rhs[s]) + heuristic_euclidean(self.start, s), min(self.g[s], self.rhs[s]))
    def update_vertex(self, u):
        if u != self.goal:
            min_rhs = float('inf')
            for succ, cost in self.graph[u].items():
                min_rhs = min(min_rhs, cost + self.g[succ])
            self.rhs[u] = min_rhs
        self.U = [item for item in self.U if item[1] != u]
        heapq.heapify(self.U)
        if self.g[u] != self.rhs[u]:
            heapq.heappush(self.U, (self.calculate_key(u), u))
            self.max_memory = max(self.max_memory, len(self.U))
    def compute_shortest_path(self):
        while self.U and (self.U[0][0] < self.calculate_key(self.start) or self.rhs[self.start] != self.g[self.start]):
            self.max_memory = max(self.max_memory, len(self.U))
            k_old, u = heapq.heappop(self.U)
            self.expanded_nodes += 1
            if k_old < self.calculate_key(u):
                heapq.heappush(self.U, (self.calculate_key(u), u))
            elif self.g[u] > self.rhs[u]:
                self.g[u] = self.rhs[u]
                for pred in self.graph[u]:
                    self.update_vertex(pred)
            else:
                self.g[u] = float('inf')
                self.update_vertex(u)
                for pred in self.graph[u]:
                    self.update_vertex(pred)


In [ ]:
# KỊCH BẢN 1: MÔI TRƯỜNG TĨNH (BEST CASE CHO A*)
res_astar_static = a_star("Arad", "Bucharest", romania_map["roads"])

start_time = time.perf_counter()
dsl = DStarLiteSimplified("Arad", "Bucharest", romania_map["roads"])
dsl.compute_shortest_path()
res_dsl_static = {
    "expanded_nodes": dsl.expanded_nodes,
    "max_memory_nodes": dsl.max_memory,
    "time_ms": (time.perf_counter() - start_time) * 1000
}

# KỊCH BẢN 2: MÔI TRƯỜNG ĐỘNG (VẬT CẢN XUẤT HIỆN Ở PITESTI)
dynamic_map = {u: {v: w for v, w in neighbors.items()} for u, neighbors in romania_map["roads"].items()}
dynamic_map["Pitesti"]["Bucharest"] = float('inf')
dynamic_map["Bucharest"]["Pitesti"] = float('inf')
res_astar_dynamic = a_star("Rimnicu Vilcea", "Bucharest", dynamic_map)

start_time = time.perf_counter()
dsl.expanded_nodes = 0
dsl.max_memory = len(dsl.U)
dsl.graph["Pitesti"]["Bucharest"] = float('inf')
dsl.graph["Bucharest"]["Pitesti"] = float('inf')
dsl.update_vertex("Pitesti")
dsl.compute_shortest_path()
res_dsl_dynamic = {
    "expanded_nodes": dsl.expanded_nodes,
    "max_memory_nodes": dsl.max_memory,
    "time_ms": (time.perf_counter() - start_time) * 1000
}

# IN RA BẢNG SO SÁNH
import pandas as pd
df = pd.DataFrame({
    "Tiêu chí": ["Expanded Nodes (Tĩnh)", "Memory - Queue Size (Tĩnh)", "Time (ms) (Tĩnh)", 
                 "Expanded Nodes (Động)", "Memory - Queue Size (Động)", "Time (ms) (Động)"],
    "A* (A-Star)": [
        res_astar_static['expanded_nodes'], res_astar_static['max_memory_nodes'], f"{res_astar_static['time_ms']:.4f}",
        res_astar_dynamic['expanded_nodes'], res_astar_dynamic['max_memory_nodes'], f"{res_astar_dynamic['time_ms']:.4f}"
    ],
    "D* Lite": [
        res_dsl_static['expanded_nodes'], res_dsl_static['max_memory_nodes'], f"{res_dsl_static['time_ms']:.4f}",
        res_dsl_dynamic['expanded_nodes'], res_dsl_dynamic['max_memory_nodes'], f"{res_dsl_dynamic['time_ms']:.4f}"
    ]
})
print("=== BẢNG SO SÁNH BENCHMARK A* VÀ D* LITE ===")
print(df.to_string(index=False))


=== BẢNG SO SÁNH BENCHMARK A* VÀ D* LITE ===
                  Tiêu chí A* (A-Star) D* Lite
     Expanded Nodes (Tĩnh)           6       5
Memory - Queue Size (Tĩnh)           6       7
          Time (ms) (Tĩnh)      0.0317  0.1431
     Expanded Nodes (Động)           6       7
Memory - Queue Size (Động)           4      10
          Time (ms) (Động)      0.0115  0.1731
